In [396]:
from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\Minas"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\Minas\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\Minas\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "UFV 5.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "UFV 5.dxf")

temp_dwg_file = os.path.join(temp_folder, "UFV 5.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)


Data Loaded


In [397]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

             start_x       start_y    rotation
count    1462.000000  1.462000e+03  1462.00000
mean   514054.791263  8.088905e+06    48.75513
std       192.126756  1.498903e+02    80.02025
min    513694.583249  8.088575e+06     0.00000
25%    513896.081249  8.088789e+06     0.00000
50%    514058.581249  8.088919e+06     0.00000
75%    514208.081249  8.089028e+06   180.00000
max    514416.083249  8.089133e+06   180.00000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      1462 non-null   object 
 1   layer     1462 non-null   object 
 2   handle    1462 non-null   object 
 3   name      1462 non-null   object 
 4   start_x   1462 non-null   float64
 5   start_y   1462 non-null   float64
 6   rotation  1462 non-null   float64
 7   UFV       1459 non-null   object 
 8   ITS       1459 non-null   object 
 9   INV       1459 non-null   object 
dt

In [398]:
string_mask = (ELM_df['name'] == "*U68") | (ELM_df['name'] == "*U67") | (ELM_df['name'] == "*U66") | (ELM_df['name'] == "*U69") | (ELM_df['name'] == "*U70")

strings_df = ELM_df[string_mask].copy()

strings_df.rename(columns={'start_x': 'Position_X', 'start_y': 'Position_Y'}, inplace=True)
strings_df.drop(columns=['name', 'rotation', 'type', 'handle', 'layer'], inplace=True)
strings_df['UFV'] = strings_df['UFV'].astype(int)
strings_df['ITS'] = strings_df['ITS'].astype(int)
strings_df['INV'] = strings_df['INV'].astype(int)
strings_df['Line'] = round(strings_df['Position_X'],0)
print(strings_df.describe())
print(strings_df.info())
print(strings_df)

          Position_X    Position_Y     UFV          ITS          INV  \
count    1378.000000  1.378000e+03  1378.0  1378.000000  1378.000000   
mean   514054.760651  8.088906e+06     5.0     1.999274    13.990566   
std       192.145953  1.527125e+02     0.0     0.816941     7.796750   
min    513694.583249  8.088575e+06     5.0     1.000000     1.000000   
25%    513896.081249  8.088789e+06     5.0     1.000000     7.000000   
50%    514055.332249  8.088919e+06     5.0     2.000000    14.000000   
75%    514208.081249  8.089028e+06     5.0     3.000000    21.000000   
max    514416.083249  8.089133e+06     5.0     3.000000    27.000000   

                Line  
count    1378.000000  
mean   514054.928157  
std       192.143710  
min    513695.000000  
25%    513896.000000  
50%    514055.500000  
75%    514208.000000  
max    514416.000000  
<class 'pandas.core.frame.DataFrame'>
Index: 1378 entries, 3 to 1380
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
--

In [399]:
inverter_mask = (ELM_df['name'] == "MDS-LAYOUT$0$PRF_Inversor") | (ELM_df['name'] == "MDS-LAYOUT$0$PRF_Inversor 18")

inverter_df = ELM_df[inverter_mask].copy()
inverter_df.rename(columns={'start_x': 'inverter_x', 'start_y': 'inverter_y'}, inplace=True)
inverter_df.drop(columns=['name', 'rotation', 'type', 'handle', 'layer'], inplace=True)
inverter_df["GRP"] = inverter_df["UFV"].astype(str) + "." + inverter_df["ITS"].astype(str) + "." + inverter_df["INV"].astype(str).str.zfill(2)
inverter_df['inverter_line'] = round(inverter_df['inverter_x'],0)
inverter_df.sort_values(by=['inverter_y', 'inverter_x'], ascending=[False, True], inplace=True)
print(inverter_df.describe())
print(inverter_df.info())
print(inverter_df)

          inverter_x    inverter_y  inverter_line
count      81.000000  8.100000e+01      81.000000
mean   514054.971022  8.088889e+06  514055.111111
std       192.825745  9.248197e+01     192.771173
min    513707.583377  8.088698e+06  513708.000000
25%    513896.085122  8.088842e+06  513896.000000
50%    514065.081122  8.088865e+06  514065.000000
75%    514208.081122  8.089009e+06  514208.000000
max    514403.081122  8.089009e+06  514403.000000
<class 'pandas.core.frame.DataFrame'>
Index: 81 entries, 1381 to 1434
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   inverter_x     81 non-null     float64
 1   inverter_y     81 non-null     float64
 2   UFV            81 non-null     object 
 3   ITS            81 non-null     object 
 4   INV            81 non-null     object 
 5   GRP            81 non-null     object 
 6   inverter_line  81 non-null     float64
dtypes: float64(3), object(4)
memory usage: 5.1+ KB


In [400]:
strings_df["GRP"] = strings_df["UFV"].astype(str) + "." + strings_df["ITS"].astype(str) + "." + strings_df["INV"].astype(str).str.zfill(2)
strings_df.sort_values(by=['Line', 'Position_Y'], ascending=[True, False], inplace=True)


#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Different PSTs
condition2 = strings_df["GRP"] != strings_df["GRP"].shift()

#Break in the Structure
condition3 = abs(strings_df["Position_Y"] - strings_df["Position_Y"].shift()) > 37

break_condition_1 = condition1 | condition2 | condition3
break_condition_2 = condition1 | condition3

strings_df['SubGroup'] = break_condition_1.cumsum()
strings_df['Group'] = break_condition_2.cumsum()
print(strings_df)
strings_df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

         Position_X    Position_Y  UFV  ITS  INV      Line     GRP  SubGroup  \
1273  513694.583249  8.088824e+06    5    1   18  513695.0  5.1.18         1   
1274  513694.583249  8.088789e+06    5    1   18  513695.0  5.1.18         1   
1275  513694.583249  8.088754e+06    5    1   18  513695.0  5.1.18         1   
1276  513694.583249  8.088719e+06    5    1   18  513695.0  5.1.18         1   
1277  513701.083249  8.088824e+06    5    1   18  513701.0  5.1.18         2   
...             ...           ...  ...  ...  ...       ...     ...       ...   
1380  514416.083249  8.089028e+06    5    3   11  514416.0  5.3.11       408   
1365  514416.083249  8.088988e+06    5    3   20  514416.0  5.3.20       409   
1366  514416.083249  8.088953e+06    5    3   20  514416.0  5.3.20       409   
1367  514416.083249  8.088919e+06    5    3   20  514416.0  5.3.20       409   
1368  514416.083249  8.088883e+06    5    3   20  514416.0  5.3.20       409   

      Group  
1273      1  
1274      1

In [401]:
distance1 = 15.9109 # Distance between the middle point of the string and the tracker's pile.

strings_df['N exit'] = strings_df['Position_Y'] + distance1
strings_df['S exit'] = strings_df['Position_Y'] - distance1

strings_df['SubGroup - Position Y - N'] = strings_df.groupby("SubGroup")["N exit"].transform('max')
strings_df['SubGroup - Position Y - S'] = strings_df.groupby("SubGroup")["S exit"].transform('min')

strings_df['Group - Position Y - N'] = strings_df.groupby("Group")["N exit"].transform('max')
strings_df['Group - Position Y - S'] = strings_df.groupby("Group")["S exit"].transform('min')

cables_df = strings_df.merge(inverter_df, on="GRP")

strings_df['No. of Strings'] = strings_df.groupby("SubGroup")['Line'].transform('count')

print(strings_df)

         Position_X    Position_Y  UFV  ITS  INV      Line     GRP  SubGroup  \
1273  513694.583249  8.088824e+06    5    1   18  513695.0  5.1.18         1   
1274  513694.583249  8.088789e+06    5    1   18  513695.0  5.1.18         1   
1275  513694.583249  8.088754e+06    5    1   18  513695.0  5.1.18         1   
1276  513694.583249  8.088719e+06    5    1   18  513695.0  5.1.18         1   
1277  513701.083249  8.088824e+06    5    1   18  513701.0  5.1.18         2   
...             ...           ...  ...  ...  ...       ...     ...       ...   
1380  514416.083249  8.089028e+06    5    3   11  514416.0  5.3.11       408   
1365  514416.083249  8.088988e+06    5    3   20  514416.0  5.3.20       409   
1366  514416.083249  8.088953e+06    5    3   20  514416.0  5.3.20       409   
1367  514416.083249  8.088919e+06    5    3   20  514416.0  5.3.20       409   
1368  514416.083249  8.088883e+06    5    3   20  514416.0  5.3.20       409   

      Group        N exit        S exit

In [402]:
groups_df = strings_df.groupby("SubGroup")[['GRP','Position_X', 'SubGroup - Position Y - N', 'SubGroup - Position Y - S', 'Group - Position Y - N', 'Group - Position Y - S', 'No. of Strings', 'Line']].first()
groups_df.rename(columns={'Position_X': 'X', 'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
print(f'Maximum number of strings in a group: {groups_df['No. of Strings'].max()}\n')
print(groups_df)

Maximum number of strings in a group: 4

             GRP              X       Y-N-STR       Y-S-STR       Y-N-TRK  \
SubGroup                                                                    
1         5.1.18  513694.583249  8.088840e+06  8.088703e+06  8.088840e+06   
2         5.1.18  513701.083249  8.088840e+06  8.088703e+06  8.088840e+06   
3         5.1.09  513707.581249  8.089004e+06  8.088972e+06  8.089004e+06   
4         5.1.18  513707.583249  8.088840e+06  8.088703e+06  8.088840e+06   
5         5.1.09  513714.081249  8.089004e+06  8.088868e+06  8.089004e+06   
...          ...            ...           ...           ...           ...   
405       5.3.20  514403.083249  8.089004e+06  8.088868e+06  8.089004e+06   
406       5.3.11  514409.583249  8.089149e+06  8.089012e+06  8.089149e+06   
407       5.3.20  514409.583249  8.089004e+06  8.088868e+06  8.089004e+06   
408       5.3.11  514416.083249  8.089149e+06  8.089012e+06  8.089149e+06   
409       5.3.20  514416.083249  8.

In [403]:
duct_map = {
    1: '-1-',
    2: '-2-',
    3: '-3-',
    4: '-4-'
}

groups_df.loc[:,'Duct Definition'] = groups_df['No. of Strings'].map(duct_map)
groups_df['Number of Ducts'] = groups_df['Duct Definition'].astype(str).str.count('-')-1
groups_df = groups_df.merge(inverter_df, on='GRP')
groups_df['N or S'] = np.where(
    abs(groups_df['Y-N-TRK'] - groups_df['inverter_y']) > abs(groups_df['Y-S-TRK'] - groups_df['inverter_y']),
    "S",
    "N"
)

groups_df['closest_y'] = np.where(
    groups_df['N or S'] == "N",
    groups_df['Y-N-TRK'],
    groups_df['Y-S-TRK']
)

groups_df.sort_values(['GRP', 'Line'], inplace = True)

print(groups_df.head(20))


       GRP              X       Y-N-STR       Y-S-STR       Y-N-TRK  \
9   5.1.01  513727.081249  8.089149e+06  8.089082e+06  8.089149e+06   
12  5.1.01  513733.581249  8.089149e+06  8.089012e+06  8.089149e+06   
15  5.1.01  513740.081249  8.089149e+06  8.089012e+06  8.089149e+06   
18  5.1.01  513746.581249  8.089149e+06  8.089012e+06  8.089149e+06   
22  5.1.01  513753.081249  8.089149e+06  8.089012e+06  8.089149e+06   
25  5.1.02  513759.581249  8.089149e+06  8.089012e+06  8.089149e+06   
28  5.1.02  513766.081249  8.089149e+06  8.089012e+06  8.089149e+06   
32  5.1.02  513772.581249  8.089149e+06  8.089012e+06  8.089149e+06   
36  5.1.02  513779.081249  8.089149e+06  8.089012e+06  8.089149e+06   
40  5.1.02  513785.581249  8.089044e+06  8.089012e+06  8.089149e+06   
39  5.1.03  513785.581249  8.089149e+06  8.089047e+06  8.089149e+06   
43  5.1.03  513792.081249  8.089149e+06  8.089012e+06  8.089149e+06   
47  5.1.03  513798.581249  8.089149e+06  8.089012e+06  8.089149e+06   
50  5.

Breaking the trenches into segments to indicate how many strings there are in each one.
=

In [404]:
"""
Adds cumulative sum column where values sum towards the inverter_line, calculated separately for each distinct group/segment.
"""
groups_df['STR_CUM_TO_INV'] = 0

# Group by GRP (or you could use 'Line Duct Definition')
# Each GRP represents a distinct physical segment with its own inverter
for grp_name in groups_df['GRP'].unique():
    # Get rows belonging to this GRP
    grp_mask = groups_df['GRP'] == grp_name
    grp_indices = groups_df[grp_mask].index

    if len(grp_indices) == 0:
        continue

    # Get the inverter_line for this GRP (should be consistent within group)
    # Using the first occurrence's inverter_line as the target for this group
    inv_line = groups_df.loc[grp_indices[0], 'inverter_line']

    # Separate rows by position relative to inverter_line
    before_inv = []
    at_inv = []
    after_inv = []

    for idx in grp_indices:
        x_val = groups_df.loc[idx, 'Line']
        if x_val < inv_line:
            before_inv.append(idx)
        elif x_val > inv_line:
            after_inv.append(idx)
        else:  # x_val == inv_line
            at_inv.append(idx)

    # Sort before_inv by X ascending (farthest to closest to inverter)
    before_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'])

    # Sort after_inv by X descending (farthest to closest to inverter)
    after_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'], reverse=True)

    # Calculate cumulative sums for before inverter (left to right)
    cumsum = 0
    for idx in before_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum

    # At inverter line: just the value itself
    for idx in at_inv:
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = groups_df.loc[idx, 'No. of Strings']

    # Calculate cumulative sums for after inverter (right to left)
    cumsum = 0
    for idx in after_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum


# Verify the results
print(groups_df[['GRP', 'X', 'No. of Strings', 'inverter_line', 'STR_CUM_TO_INV']].to_string())
max_number_of_strings = groups_df['STR_CUM_TO_INV'].max()
groups_df.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")

        GRP              X  No. of Strings  inverter_line  STR_CUM_TO_INV
9    5.1.01  513727.081249               2       513740.0               2
12   5.1.01  513733.581249               4       513740.0               6
15   5.1.01  513740.081249               4       513740.0               4
18   5.1.01  513746.581249               4       513740.0               8
22   5.1.01  513753.081249               4       513740.0               4
25   5.1.02  513759.581249               4       513773.0               4
28   5.1.02  513766.081249               4       513773.0               8
32   5.1.02  513772.581249               4       513773.0               4
36   5.1.02  513779.081249               4       513773.0               5
40   5.1.02  513785.581249               1       513773.0               1
39   5.1.03  513785.581249               3       513799.0               3
43   5.1.03  513792.081249               4       513799.0               7
47   5.1.03  513798.581249            

In [405]:
distance2 = 0.8805 # Distance between the first/last pile and the first/last module

cables_df.drop(columns=['UFV_x', 'UFV_y', 'ITS_x', 'ITS_y', 'INV_x', 'INV_y', 'SubGroup', 'Group'], inplace=True)
cables_df.rename(columns={'Position_X': 'X', 'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
cables_df['N or S'] = np.where(
    abs(cables_df['N exit'] - cables_df['inverter_y']) > abs(cables_df['S exit'] - cables_df['inverter_y']),
    "S",
    "N"
)

cables_df['closest_y'] = np.where(
    cables_df['N or S'] == "N",
    cables_df['Y-N-TRK'],
    cables_df['Y-S-TRK']
)

cables_df['Y-N-STR'] = cables_df['Y-N-STR'] + distance2
cables_df['Y-S-STR'] = cables_df['Y-S-STR'] - distance2

cables_df.sort_values(['GRP', 'Line'], inplace = True)
print(cables_df.head())

                X    Position_Y      Line     GRP        N exit        S exit  \
29  513727.081249  8.089133e+06  513727.0  5.1.01  8.089149e+06  8.089117e+06   
30  513727.081249  8.089098e+06  513727.0  5.1.01  8.089114e+06  8.089082e+06   
39  513733.581249  8.089133e+06  513734.0  5.1.01  8.089149e+06  8.089117e+06   
40  513733.581249  8.089098e+06  513734.0  5.1.01  8.089114e+06  8.089082e+06   
41  513733.581249  8.089063e+06  513734.0  5.1.01  8.089079e+06  8.089047e+06   

         Y-N-STR       Y-S-STR       Y-N-TRK       Y-S-TRK     inverter_x  \
29  8.089149e+06  8.089081e+06  8.089149e+06  8.089082e+06  513740.081122   
30  8.089149e+06  8.089081e+06  8.089149e+06  8.089082e+06  513740.081122   
39  8.089149e+06  8.089011e+06  8.089149e+06  8.089012e+06  513740.081122   
40  8.089149e+06  8.089011e+06  8.089149e+06  8.089012e+06  513740.081122   
41  8.089149e+06  8.089011e+06  8.089149e+06  8.089012e+06  513740.081122   

      inverter_y  inverter_line N or S     closest

In [406]:
script_df = groups_df[['No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_y', 'X', 'inverter_x', 'inverter_y','Line', 'inverter_line', 'STR_CUM_TO_INV', 'Y-N-STR', 'Y-S-STR', 'N or S']].copy()
script_df.rename(columns={'closest_y': 'Y'}, inplace=True)

distance3 = 2.634 # Vertical distance between the tracker's pile and the inverter.
distance4 = 0.675 # Distance between the middle point of the inverter and its edge on the X axis.
distance5 = 0.1825 # Distance between the middle point of the inverter and its edge on the Y axis.

def intermediary_coordinates(df):
    df['intermediary_x'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['X'] + distance3,
            df['X'] - distance3
        )
    )

    df['inverter_x'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance4,
            df['inverter_x'] + distance4
        )
    )

    df['inverter_y'] = np.where(
        df['inverter_line'] != df['Line'],
        df['inverter_y'],
        np.where(
            df['N or S'] == 'S',
            df['inverter_y'] + distance5,
            df['inverter_y'] - distance5
        )
    )

'''

CABOS

'''

intermediary_coordinates(cables_df)

def calculate_cable_length(row, color):
    try:
        x = row['X']
        y_n_str = row['Y-N-STR']
        y_s_str = row['Y-S-STR']
        closest_y = row['closest_y']
        int_x = row['intermediary_x']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']

        if color == 'red':
            seg1 = math.sqrt((x - x)**2 + (y_n_str - closest_y)**2)
        else:
            seg1 = math.sqrt((x - x)**2 + (y_s_str - closest_y)**2)
        seg2 = math.sqrt((x - int_x)**2 + (closest_y - inv_y)**2)
        seg3 = math.sqrt((int_x - inv_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2 + seg3
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise

cables_df['Red Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'red'), axis=1)
cables_df['Black Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'black'), axis=1)

total_red_length = cables_df['Red Cable Length'].sum()
total_black_length = cables_df['Black Cable Length'].sum()

total_cable_length = total_red_length + total_black_length

cables_df['Script Red Cables'] = cables_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y-N-STR']} {row['X']},{row['closest_y']} {row['intermediary_x']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
cables_df['Script Black Cables'] = cables_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y-S-STR']} {row['X']},{row['closest_y']} {row['intermediary_x']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Cabos Tracker.scr", 'w') as f:
    f.write('PLINEWID 0\n')
    f.write("LAYER S EMF_cabo_solar_positivo\n\n")
    f.write('\n'.join(cables_df["Script Red Cables"].astype(str)) + '\n')
    f.write("LAYER S EMF_cabo_solar_negativo\n\n")
    f.write('\n'.join(cables_df["Script Black Cables"].astype(str)) + '\n')

'''

VALAS

'''

intermediary_coordinates(script_df)

script_df['Script Valas'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
print(script_df['Script Valas'])

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Valas Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    f.write("LAYER S EMF_bt_vala\n\n")
    f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')


'''

DUTOS

'''

def calculate_duct_length(row, no_of_strings):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']
        number_of_ducts = row['Duct Definition'].count(no_of_strings)

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - inv_x)**2 + (inv_y - inv_y)**2)

        return (seg1 + seg2) * number_of_ducts
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise

script_df['Dutos 1 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '1'), axis=1)
script_df['Dutos 2 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '2'), axis=1)
script_df['Dutos 3 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '3'), axis=1)
script_df['Dutos 4 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '4'), axis=1)

total_1str_duct_length = script_df['Dutos 1 Length'].sum()
total_2str_duct_length = script_df['Dutos 2 Length'].sum()
total_3str_duct_length = script_df['Dutos 3 Length'].sum()
total_4str_duct_length = script_df['Dutos 4 Length'].sum()

total_duct_length = total_1str_duct_length + total_2str_duct_length + total_3str_duct_length + total_4str_duct_length

script_df['Script Dutos 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('1'),
    axis=1
)
script_df['Script Dutos 2'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('2'),
    axis=1
)
script_df['Script Dutos 3'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('3'),
    axis=1
)
script_df['Script Dutos 4'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('4'),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Dutos Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.03\n')
    # Layer 1
    layer1_commands = script_df["Script Dutos 1"].astype(str)
    layer1_commands = layer1_commands[layer1_commands != '']  # Remove empty strings
    if len(layer1_commands) > 0:  # Only write layer if there are commands
        f.write("LAYER S EMF_bt_eletroduto_1S\n\n")
        f.write('\n'.join(layer1_commands) + '\n')

    # Layer 2
    layer2_commands = script_df["Script Dutos 2"].astype(str)
    layer2_commands = layer2_commands[layer2_commands != '']
    if len(layer2_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_2S\n\n")
        f.write('\n'.join(layer2_commands) + '\n')

    # Layer 3
    layer3_commands = script_df["Script Dutos 3"].astype(str)
    layer3_commands = layer3_commands[layer3_commands != '']
    if len(layer3_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_3S\n\n")
        f.write('\n'.join(layer3_commands) + '\n')

    # Layer 4
    layer4_commands = script_df["Script Dutos 4"].astype(str)
    layer4_commands = layer4_commands[layer4_commands != '']
    if len(layer4_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_4S\n\n")
        f.write('\n'.join(layer4_commands) + '\n')

9      _.pline 513727.0812493581,8089081.9604809275 5...
12     _.pline 513733.5812493581,8089011.97575535 513...
15     _.pline 513740.0812493581,8089011.97575535 513...
18     _.pline 513746.5812493581,8089011.97575535 513...
22     _.pline 513753.0812493581,8089011.97575535 513...
                             ...                        
369    _.pline 514325.0812493581,8088839.638330604 51...
373    _.pline 514331.5812493581,8088839.638330604 51...
378    _.pline 514338.0812493581,8088839.638330604 51...
381    _.pline 514344.5812493581,8088839.638330604 51...
384    _.pline 514351.0812493581,8088839.638330604 51...
Name: Script Valas, Length: 409, dtype: object


In [407]:
'''

LAYERED VALAS

'''

pitch = 6.5

script_df['intermediary_x_2'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df['intermediary_x'] + pitch,
        script_df['intermediary_x'] - pitch
    )
)

script_df['final_x'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df.apply(lambda row: min(row['intermediary_x_2'], row['inverter_x']), axis=1),
        script_df.apply(lambda row: max(row['intermediary_x_2'], row['inverter_x']), axis=1)
    )
)

def calculate_trench_length(row):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x']
        inv_y = row['inverter_y']
        final_x = row['final_x']

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - final_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise


script_df['Valas Length'] = script_df.apply(calculate_trench_length, axis=1)

total_trench_length = script_df['Valas Length'].sum()

script_df['Script Valas Layered 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x']},{row['inverter_y']} ",
    axis=1
)

script_df['Script Valas Layered 2'] = script_df.apply(
    lambda row: f"_.pline {row['intermediary_x']},{row['inverter_y']} {row['final_x']},{row['inverter_y']} ",
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Valas Layered Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    for i in range(1, max_number_of_strings+1):
        mask1 = script_df['No. of Strings'] == i
        mask2 = script_df['STR_CUM_TO_INV'] == i

        if mask1.any() or mask2.any():
            f.write(f"LAYER S EMF_vala_CC_{i}C\n\n")

            if mask1.any():
                f.write('\n'.join(script_df[mask1]["Script Valas Layered 1"].astype(str)) + '\n')

            if mask2.any():
                f.write('\n'.join(script_df[mask2]["Script Valas Layered 2"].astype(str)) + '\n')


lengths_df = pd.DataFrame.from_dict(data={
    'DC Red Cable Length': [total_red_length],
    'DC Black Cable Length': [total_black_length],
    'DC Total Cable Length': [total_cable_length],
    '1str Duct Length': [total_1str_duct_length],
    '2str Duct Length': [total_2str_duct_length],
    '3str Duct Length': [total_3str_duct_length],
    '4str Duct Length': [total_4str_duct_length],
    'Total Duct Length': [total_duct_length],
    'Total Trench Length': [total_trench_length]
}, orient="index", columns=['Length (m)'])
lengths_df.to_excel(r"C:\Users\Usuario\Desktop\Lengths.xlsx")